# Naver 뉴스 URL 수집 — 언론사+기간 모드 (Colab용)

네이버 뉴스의 언론사별 일자 페이지(`news.naver.com/main/list.naver?mode=LPOD&...`) 를 순회하며 기사 URL 수집 — 통신3사 트랙과 동일하게 **기간 단위 통합 JSON** 1개 산출, 내부에서 일자별 페이지 순회하며 임시 체크포인트로 중단·재개 지원. 리스트 페이지가 정적 HTML이라 `requests` + `BeautifulSoup` 만 사용

- 입력 — `press_ranges` (언론사 `press`·`oid` + 기간 `start_date`·`end_date`)
- 출력 — `링크_{press}_{YYMMDD}_{YYMMDD}.json` (기간 통합 기사 URL)
- 보조 출력 — 기간별 수집 로그 JSON, 중간 재개용 temp JSON
- 특징 — 기간 단위 통합 저장 · 일자별 페이지 순회 · 임시 파일로 중단 재개 · max page 자동 감지 · EUC-KR 인코딩 처리 · 중복 제거 · 완료 파일 건너뛰기


In [16]:
# Colab 환경 세팅 — requests, BeautifulSoup 설치 (Selenium 불필요)
# !pip install -q requests beautifulsoup4


In [17]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Drive 안의 프로젝트 폴더로 이동
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')


현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring


In [19]:
import json
import os
import time
from datetime import datetime, timedelta
from pathlib import Path

import requests
from bs4 import BeautifulSoup

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 언론사와 수집 기간 지정
# press 하나당 start_date ~ end_date 안의 일자를 한 통합 JSON으로 묶어 저장 (통신3사 트랙과 동일 패턴)
# 날짜 형식: 'YYYY.MM.DD'
# oid는 네이버 언론사 식별자: SBS=055, KBS=056, MBC=214 (LPOD 페이지 URL의 oid 파라미터로 사용)
press_ranges = [
    {'press': 'SBS', 'oid': '055', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'KBS', 'oid': '056', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'MBC', 'oid': '214', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '한국경제', 'oid': '015', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '매일경제', 'oid': '009', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '한겨레', 'oid': '028', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '조선일보', 'oid': '023', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '연합뉴스', 'oid': '001', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'YTN',     'oid': '052', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]


# 기간 단위 작업 목록 생성 — press_ranges 한 항목당 jobs 1개
# 내부 일자 순회는 collect_links_for_period에서 처리하므로 일자별 분할 jobs는 만들지 않음
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 문자열을 datetime으로 파싱 — 기간 유효성 검사용
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        jobs.append({
            'press': item['press'],
            'oid': item['oid'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 생성된 jobs는 다음 셀에서 순서대로 실행
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'

# 저장할 폴더 지정 — 링크 파일, 수집 로그, 임시 체크포인트, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = Path(PROJECT_DIR) / 'news' / 'notebook' / 'crawling' / 'data'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

# requests 세션 생성 — User-Agent / 언어 헤더를 반복 요청에 적용
# Selenium을 안 쓰는 이유: LPOD 리스트 페이지는 정적 HTML이라 requests로 충분, 속도도 훨씬 빠름
session = requests.Session()
session.headers.update({
    'User-Agent': USER_AGENT,
    # 한국어 우선 응답 유도 (네이버 일부 페이지가 Accept-Language로 분기)
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
})
print(f'User-Agent: {USER_AGENT}')


총 작업 수: 9
{'press': 'SBS', 'oid': '055', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': 'KBS', 'oid': '056', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': 'MBC', 'oid': '214', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': '한국경제', 'oid': '015', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': '매일경제', 'oid': '009', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': '한겨레', 'oid': '028', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': '조선일보', 'oid': '023', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': '연합뉴스', 'oid': '001', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': 'YTN', 'oid': '052', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
저장 위치: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36


In [20]:
import random
import re

# 서버 부담을 줄이기 위해 페이지/일자/job 사이에 랜덤 대기
# PAGE_PAUSE: 같은 일자 내 페이지 이동 (짧게)
# DAY_PAUSE: 같은 기간 내 일자 전환 (중간) — 통신3사 트랙(url_수집_colab.ipynb)과 동일 값
# JOB_PAUSE: 다른 언론사로 전환 (길게)
PAGE_PAUSE_RANGE_SEC = (0.4, 1.2)
DAY_PAUSE_RANGE_SEC = (2, 5)
JOB_PAUSE_RANGE_SEC = (5, 12)
REQUEST_TIMEOUT_SEC = 10  # requests 응답 대기 상한 — 네트워크 멈춤 시 무한 hang 방지
SKIP_COMPLETED = True  # 최종 링크 파일이 이미 있는 기간은 건너뜀
USE_YEONHAP_EXTENDED_URL = False  # 연합뉴스(oid=001) LPOD URL에 sid1/sid2/isYeonhapFlash 추가형 사용 — 검증 후 True로 토글


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


def build_list_url(oid, date_ymd, page=1):
    # 네이버 언론사 일자별 리스트 URL — date_ymd 형식: 'YYYYMMDD'
    # mode=LPOD: 언론사별 송고 기사 리스트, mid=sec: 섹션 분류 무관
    # 연합뉴스(oid=001)는 LPOD가 sid1/sid2/isYeonhapFlash 가 붙는 확장형으로 서빙됨 — 검증 후 USE_YEONHAP_EXTENDED_URL=True 토글
    if USE_YEONHAP_EXTENDED_URL and oid == '001':
        return (
            f'https://news.naver.com/main/list.naver?'
            f'mode=LPOD&sid2=140&sid1=001&mid=sec'
            f'&oid={oid}&isYeonhapFlash=Y&date={date_ymd}&page={page}'
        )
    return (
        f'https://news.naver.com/main/list.naver?mode=LPOD&mid=sec'
        f'&oid={oid}&date={date_ymd}&page={page}'
    )


def fetch_list_html(oid, date_ymd, page=1):
    # 네이버 언론사 리스트 페이지는 EUC-KR이라 인코딩 명시 필요
    # requests가 Content-Type 헤더만 보고 latin-1로 잘못 추정하는 경우가 있어 강제 지정
    response = session.get(build_list_url(oid, date_ymd, page), timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()
    response.encoding = 'euc-kr'
    return response.text


def detect_max_page(html, oid, date_ymd):
    # 페이지네이션 영역에 page=N 링크들이 노출됨 → 그중 max 값이 마지막 페이지
    # 부산일보식 enumerate 순회 대신, 페이지바를 한 번에 파싱해 끝 페이지 확정
    pattern = re.compile(rf'oid={oid}[^"\s]*?date={date_ymd}[^"\s]*?page=(\d+)')  # 확장 URL(연합뉴스) 호환
    pages = [int(m) for m in pattern.findall(html)]
    return max(pages) if pages else 1  # 매칭 실패 시 1로 폴백 (단일 페이지일 경우)


def extract_article_links(html, oid):
    # 페이지에는 헤드라인 강조용 ul.type06_headline 과 일반 ul.type06 두 개가 존재
    # 둘 모두에서 dt > a 의 href를 읽어 n.news.naver.com/mnews/article/{oid}/{id} 만 필터
    soup = BeautifulSoup(html, 'html.parser')
    links = set()
    # 같은 언론사(oid) 기사만 모으도록 정규식에 oid 박아둠 — 외부 링크/공유 링크 제외
    article_path_pattern = re.compile(rf'^https://n\.news\.naver\.com/mnews/article/{oid}/\d+')
    for a in soup.select('ul.type06_headline dt a, ul.type06 dt a'):
        href = a.get('href', '')
        if article_path_pattern.match(href):
            # ?sid=... query string 떼고 본 URL만 보관 (sid 차이로 인한 중복 차단)
            links.add(href.split('?')[0])
    return links


# 한 일자의 모든 페이지를 순회하며 기사 링크 set 반환 + 페이지별 통계
def collect_links_for_day(oid, date_ymd):
    # 1페이지 받기 + max page 감지 — 첫 응답은 아래 루프에서 재활용
    first_html = fetch_list_html(oid, date_ymd, page=1)
    max_page = detect_max_page(first_html, oid, date_ymd)

    day_links = set()
    page_stats = []

    # 1 ~ max_page 순회하며 기사 링크 누적
    for page in range(1, max_page + 1):
        if page == 1:
            html = first_html  # 위에서 받아둔 1페이지 HTML 재사용
        else:
            # 2페이지부터는 페이지 전환 대기 후 새로 요청
            polite_sleep(f"  page {page} 받기 전", PAGE_PAUSE_RANGE_SEC)
            html = fetch_list_html(oid, date_ymd, page=page)

        page_links = extract_article_links(html, oid)
        before = len(day_links)
        day_links.update(page_links)
        # 페이지별 수집량 로그 — 특정 페이지에서 0건이 나오면 셀렉터 문제 의심
        page_stats.append({
            'page': page,
            'found': len(page_links),
            'added': len(day_links) - before,
            'total_this_day': len(day_links),
        })

    return day_links, max_page, page_stats


# 한 언론사의 기간 전체를 통합 JSON 1개로 저장 (통신3사 collect_links와 동일 패턴)
# 일자별 임시 체크포인트로 중단/재개 지원
def collect_links_for_period(press, oid, start_date, end_date, save_dir=SAVE_DIR):
    # 파일명에 들어가는 기간 접미사 (예: 2026.05.01~2026.05.07 -> 260501_260507)
    start_yymmdd = start_date.replace('.', '')[2:]
    end_yymmdd = end_date.replace('.', '')[2:]
    period = f"{start_yymmdd}_{end_yymmdd}"

    # 파일 경로 — temp: 일자 단위 체크포인트 / links: 최종 통합 / stats: 수집 로그
    temp_links_path = save_dir / f"{press}_{start_date}_{end_date}_temp_links.json"
    links_save_path = save_dir / f"링크_{press}_{period}.json"
    stats_save_path = save_dir / f"수집로그_{press}_{period}.json"

    # 최종 파일이 이미 있으면 같은 기간은 건너뜀 (밤새 재시작에도 idempotent)
    if SKIP_COMPLETED and links_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {links_save_path}")
        return links_save_path

    print()
    print(f"=== {press} / {start_date} ~ {end_date} 수집 시작 ===")

    # 임시 파일에 기존 링크가 있으면 불러오기 — last_date 다음 날부터 이어서 수집
    if temp_links_path.exists():
        with temp_links_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # set으로 변환해 이미 모은 링크와 신규 링크 중복 차단
        all_links_set = set(checkpoint.get('links', []))
        last_collected_date = checkpoint.get('last_date')
        # daily_stats도 임시 파일에 같이 보관해서 중단 전 로그 유지
        daily_stats = checkpoint.get('daily_stats', [])
        print(f"기존 임시 파일에서 링크 {len(all_links_set)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집")
    else:
        all_links_set = set()
        last_collected_date = None
        daily_stats = []
        print('새로 링크 수집 시작')

    # 시작 ~ 끝 일자를 하루씩 순회
    current = datetime.strptime(start_date, '%Y.%m.%d')
    end = datetime.strptime(end_date, '%Y.%m.%d')

    while current <= end:
        day_str = current.strftime('%Y.%m.%d')

        # 이미 수집 완료한 날짜면 건너뜀 (재시작 시 중복 fetch 차단)
        if last_collected_date and day_str <= last_collected_date:
            print(f"{day_str} — 이미 수집 완료, 건너뜀")
            current += timedelta(days=1)
            continue

        date_ymd = day_str.replace('.', '')  # 'YYYYMMDD' — URL 파라미터용
        started_at = time.time()

        # 하루치 모든 페이지 순회해서 링크 추출
        day_links, max_page, page_stats = collect_links_for_day(oid, date_ymd)
        before = len(all_links_set)
        all_links_set.update(day_links)
        added = len(all_links_set) - before  # 일자 간 중복 제거 후 늘어난 개수
        elapsed = round(time.time() - started_at, 2)

        # 일자별 수집량/페이지 수/소요 시간 로그 — 사후 진단용
        daily_stats.append({
            'date': day_str,
            'found': len(day_links),
            'added': added,
            'total': len(all_links_set),
            'max_page': max_page,
            'elapsed_sec': elapsed,
            'pages': page_stats,
        })

        print(
            f"{day_str} — {len(day_links)}건 수집 / 신규 {added}건 추가 "
            f"/ 누적 {len(all_links_set)}건 / 페이지 {max_page} / {elapsed}초"
        )

        # 하루치 수집 후 임시 파일에 즉시 저장 (중간에 끊겨도 누적 보존 + 마지막 완료 날짜 기록)
        with temp_links_path.open('w', encoding='utf-8') as f:
            json.dump(
                {'links': sorted(all_links_set), 'last_date': day_str, 'daily_stats': daily_stats},
                f, ensure_ascii=False, indent=2,
            )
        last_collected_date = day_str
        current += timedelta(days=1)
        # 다음 날짜로 넘어가기 전 대기 (마지막 날 뒤엔 안 함)
        if current <= end:
            polite_sleep('다음 날짜 전', DAY_PAUSE_RANGE_SEC)

    # 기간 통합 최종 저장 — sorted로 안정적 순서 (diff/재현성)
    naver_news_links = sorted(all_links_set)
    with open(links_save_path, 'w', encoding='utf-8') as f:
        json.dump(naver_news_links, f, ensure_ascii=False, indent=2)

    # 수집 로그 — 일자별 통계 + 전체 요약
    with open(stats_save_path, 'w', encoding='utf-8') as f:
        json.dump({
            'press': press,
            'oid': oid,
            'start_date': start_date,
            'end_date': end_date,
            'total': len(naver_news_links),
            'days': daily_stats,
        }, f, ensure_ascii=False, indent=2)

    # 정상 완료 시 임시 파일 삭제 — 다음 실행에서 그릇된 재개 방지
    if temp_links_path.exists():
        temp_links_path.unlink()

    print(f"수집 완료 — 총 {len(naver_news_links)}개")
    print(f"링크 저장: {links_save_path}")
    print(f"수집 로그 저장: {stats_save_path}")
    return links_save_path


# job 단위로 실행 (job 하나 = 언론사 × 기간)
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # job 딕셔너리의 press/oid/start_date/end_date를 collect_links_for_period 인자로 전달
        results.append(collect_links_for_period(**job))
    except Exception as exc:
        # 한 언론사에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어갑니다: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 job(언론사)으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = SAVE_DIR / '수집실패목록_naver.json'
    with open(failures_path, 'w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)



[1/9] 작업 실행: {'press': 'SBS', 'oid': '055', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}

=== SBS / 2026.05.05 ~ 2026.05.11 이미 완료됨, 건너뜀 ===
기존 파일: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data/링크_SBS_260505_260511.json
다음 작업 전 7.9초 대기

[2/9] 작업 실행: {'press': 'KBS', 'oid': '056', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}

=== KBS / 2026.05.05 ~ 2026.05.11 이미 완료됨, 건너뜀 ===
기존 파일: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data/링크_KBS_260505_260511.json
다음 작업 전 5.0초 대기

[3/9] 작업 실행: {'press': 'MBC', 'oid': '214', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}

=== MBC / 2026.05.05 ~ 2026.05.11 이미 완료됨, 건너뜀 ===
기존 파일: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data/링크_MBC_260505_260511.json
다음 작업 전 5.2초 대기

[4/9] 작업 실행: {'press': '한국경제', 'oid': '015', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}

=== 한국경제 / 2026.05.05 ~ 2026.05.11 이미 완료됨, 건너뜀 ===
기존 파일: /con